<a href="https://colab.research.google.com/github/Naveed-Bhutto/Assignment_PIAIC/blob/main/Project2_RAG_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -qU langchain-pinecone langchain-google-genai langchain-community python-dotenv

<ins>**Project2 version 1** </ins>\
RAG using data search inputby user with pinecone database using LangChain and LLM model="gemini-2.0-flash-exp"

In [ ]:
import getpass
import os
import time

from pinecone import Pinecone, ServerlessSpec

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter your Pinecone API key: ")

pinecone_api_key = os.environ.get("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

Enter your Pinecone API key: ··········


In [ ]:
# Code was Ok
index_name = "online-project2-rag2"

pc.create_index(
    name=index_name,
    dimension=768,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)
index = pc.Index(index_name)

In [ ]:

# Delete the existing index
pc.delete_index(index_name)
print(f"Index '{index_name}' deleted.")
# Wait for deletion to complete
time.sleep(10)  # Adjust wait time as needed

# Recreate the index with the new dimension
pc.create_index(
    name=index_name,
    dimension=768,  # Updated dimension
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)
index = pc.Index(index_name)
print(f"Index '{index_name}' ") # recreated with dimension 768.


Index 'online-project2-rag2' deleted.
Index 'online-project2-rag2' 


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [ ]:
vector = embeddings.embed_query("hello, world!")
vector[:5]

[0.05168594419956207,
 -0.030764883384108543,
 -0.03062233328819275,
 -0.02802734263241291,
 0.01813093200325966]

In [ ]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index , embedding=embeddings)

In [ ]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had a superb evening tea with delicious cookies.",
    metadata={"source": "Facebook post"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow in Karachi is cold, with a low temperature of 11 degrees.",
    metadata={"source": "Geo news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain, pinecone and RAF- come check it out!",
    metadata={"source": "WhatsApp"},
)

document_4 = Document(
    page_content="Robbers broke into the Korangi Karachi area in Allied Bank and stole PKR 1 million in cash.",
    metadata={"source": "Geo news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "WhatsApp"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "WhatsApp"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "Geo news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "WhatsApp"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [ ]:
from uuid import uuid4
uuid4()

UUID('86d7300e-1d71-4692-8822-a955d21bd298')

In [ ]:
# Data save

uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['137e7a74-628a-4cbf-a95b-050d40b0efba',
 '08146667-db18-4026-a6d2-eefefdd0a8ad',
 'efbbf621-70f6-4324-b452-c9df2e5e956d',
 '60a3b8ab-9883-4873-98a0-0b9549101d0a',
 '3a1fc292-59b0-47f1-bc2d-1c72cc49dac6',
 'ee475b45-d05b-42d7-93da-1b1256c8f87d',
 '4cdedd91-4ef7-4457-b8a5-7ea00affeb13',
 '78992c68-bb0d-46fd-ab91-6c47d4e893f5',
 '789f4719-34e4-4bd2-83e0-fa4173d7fd22',
 'ff7ff712-fc72-4df7-a76a-c42d65712f4b']

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp")

In [ ]:
from langchain.schema import HumanMessage

def answer_to_user(query: str):

    # Vector search
    results = vector_store.similarity_search_with_score(query)

    # Prepare context for the LLM
    context = ""
    for res, score in results:
        context += f"[SIM={score:3f}] {res.page_content} [{res.metadata}]\n"

    # Construct the prompt for the LLM
    prompt = f"Context:\n{context}\n Question:\n{query}\n Answer:"
    # prompt = f"Context:\n{context}\n\nQuestion:\n{query}\n\nAnswer:"

    # Pass the prompt to the LLM
    final_answer = model([HumanMessage(content=prompt)]) # Assuming the model can take a string prompt

    return final_answer

# Example usage

while True:
    user_query = input("Enter your query: ")
    if user_query.lower() == "exit":
        break
    response = answer_to_user(user_query)
    print(response)

<ins>**Project2 version 2.0** </ins>\
RAG Project using data search within uploaded PDF documents with input by user. pinecone database using LangChain and LLM model="gemini-1.5-flash"

In [31]:
%pip install -qU langchain-pinecone langchain-google-genai langchain-community python-dotenv pypdf pyPDF2 google-generativeai streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 9.2 MB/s eta 0:00:00


In [39]:
import streamlit as st
import os
import pinecone
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import google.generativeai as genai

from langchain.vectorstores import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains.question_answering import load_qa_chain
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [40]:
import getpass
import os
import time

from pinecone import Pinecone, ServerlessSpec

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter your Pinecone API key: ")

pinecone_api_key = os.environ.get("PINECONE_API_KEY")
PINECONE_ENVIRONMENT = os.getenv("PINECONE_ENVIRONMENT")  # e.g., "gcp-starter"

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
# Code for Project2 version 2.0
index_name = "online-project2-rag2-v2"

pc.create_index(
    name=index_name,
    dimension=768,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)
index = pc.Index(index_name)

In [41]:
# Delete the existing index
pc.delete_index(index_name)
print(f"Index '{index_name}' deleted.")
# Wait for deletion to complete
time.sleep(10)  # Adjust wait time as needed

# Recreate the index with the new dimension
pc.create_index(
    name=index_name,
    dimension=768,  # Updated dimension
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)
index = pc.Index(index_name)
print(f"Index '{index_name}' ") # recreated with dimension 768.


Index 'online-project2-rag2-v2' deleted.
Index 'online-project2-rag2-v2' 


In [42]:
from langchain.document_loaders import PyPDFLoader, PyPDFDirectoryLoader
from uuid import uuid4
def load_and_process_document(file_path):
    """Loads a document and processes it for embedding and storage."""
    file_extension = os.path.splitext(file_path)[1].lower()

    if file_extension == ".pdf":
        loader = PyPDFLoader(file_path)
    else:
        print(f"Unsupported file type: {file_extension}")
        return None

    try:
        documents = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
        docs = text_splitter.split_documents(documents)
        return docs
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return None

In [43]:
from langchain_pinecone import PineconeVectorStore
vector = embeddings.embed_query("hello, world!")
vector_store = PineconeVectorStore(index=index , embedding=embeddings)


In [44]:
file_paths = ["/content/Document1.pdf", "/content/Document2.pdf"]  # Replace with your file paths

all_documents = []
for file_path in file_paths:
    docs = load_and_process_document(file_path)
    if docs:
        all_documents.extend(docs)

# Add the documents to Pinecone (if any were successfully loaded and processed)
if all_documents:
    uuids = [str(uuid4()) for _ in range(len(all_documents))]
    vector_store.add_documents(documents=all_documents, ids=uuids)
    print(f"Added {len(all_documents)} chunks to Pinecone.")
else:
    print("No documents were successfully loaded and processed.")

Added 68 chunks to Pinecone.


In [55]:
def get_pdf_text(pdf_docs):
    text = ""
    for pdf in pdf_docs:
        pdf_reader = PdfReader(pdf)
        for page in pdf_reader.pages:
            text += page.extract_text()
    return text

def get_text_chunks(text):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=10000,
        chunk_overlap=1000,
        length_function=len
    )
    chunks = text_splitter.split_text(text)
    return chunks

def get_vector_store(text_chunks):
    embeddings = GoogleGenerativeAIEmbeddings(model="model/embedding-001")
    vector_store = Pinecone(index_name, embeddings.embed_query)
    vector_store = Pinecone(text_chunks, embeddings.embed_query)


    vector_store.add_texts("pinecone_index")

def get_conversational_chain(vector_store):
    prompt_template = f"""
    Answer the question as detailed as possible from the provided context make sure to provide all the details,
    if the answer is not in provided context just say, "answer is not available in the context", don't provide the wrong answer.\n\n
    Context:\n {context}?\n
    Question: \n {question}\n

    Answer:"""

    model = ChatGoogleGenerativeAI(model="gemini-1.5-flash",temperature=0.3)

    prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
    chain = load_qa_chain(model, chain_type="stuff", prompt=prompt)
    return chain


def user_input(user_question):
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
    new_db = Pinecone(index_name, embeddings.embed_query)
    docs = new_db.similarity_search(user_question)

    chain = get_conversational_chain()


    response = chain(
        {"input_documents": docs, "question": user_question}
        , return_only_outputs=True
        )
    print(response)

In [ ]:
from langchain.schema import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-1.5-flash",temperature=0.3)

def answer_to_user(query: str):

    # Vector search
    results = vector_store.similarity_search_with_score(query)

    # Prepare context for the LLM
    context = ""
    for res, score in results:
        context += f"[SIM={score:3f}] {res.page_content} [{res.metadata}]\n"

    # Construct the prompt for the LLM
    prompt = f"Context:\n{context}\n\nQuestion:\n{query}\n\nAnswer:"

    # Pass the prompt to the LLM
    final_answer = model([HumanMessage(content=prompt)]) # Assuming the model can take a string prompt

    return final_answer

# Example usage

while True:
    user_query = input("Enter your query: ")
    if user_query.lower() == "exit":
        break
    response = answer_to_user(user_query)
    print(response)